# VRPTW（Solomon c101）—— 拉格朗日松弛

## 问题定义

集合覆盖模型（同 02）：$$\min \sum_{p\in P} c_p x_p \quad \text{s.t.}\quad \sum_p a_{ip}x_p \ge 1\ (\forall i),\ \sum_p x_p \le K,\ x_p\in\{0,1\}$$

## 方法：松弛覆盖约束的对偶分解

**松弛** 25 个覆盖约束（乘子 $\lambda_i\ge 0$），得对偶函数：

$$L(\lambda) = \min_{\Sigma x \le K,\ x\in\{0,1\}} \sum_p c_p x_p + \sum_i \lambda_i\Big(1-\sum_p a_{ip}x_p\Big)
= \sum_i \lambda_i + \min_{\Sigma x \le K} \sum_p \underbrace{\Big(c_p - \sum_i a_{ip}\lambda_i\Big)}_{rc_p} x_p$$

**子问题（平凡，无需求解器）**：按 $rc_p$ 排序，取最负的至多 $K$ 个列（$rc\ge 0$ 不取）。
子问题的 LP 松弛与整数解同值（**integrality property**）⇒ 拉格朗日对偶 $\max_{\lambda\ge 0} L(\lambda)$
= 集合覆盖 **LP 松弛值** = 191.813620（02 已证）。

**次梯度法**（600 轮）：$g_i = 1 - \sum_p a_{ip} x_p(\lambda)$（注意是**覆盖次数**而非布尔标志）；
$$\lambda_i \leftarrow \max\big(0,\ \lambda_i + \alpha\, g_i\big),\qquad \alpha = \rho\,\frac{UB - L(\lambda)}{\|g\|^2}$$

$\rho=2.0$ 起、30 轮无下界改进减半；步长截断 100、$\lambda$ 截断 1000（防发散）；$UB$ 锚点取已知上界（单客户往返总成本 1132.2）。

**原始解修复**：① 每轮对子问题解贪心补列（给未覆盖客户加最便宜覆盖列，≤K 列）得可行上界；
② 结束后在候选池上解 CP-SAT 集合覆盖整数模型（MIP 修复）。


## 实现要点

1. **候选列集**：完整池 210,449 列可枚举但每轮全池扫描过重；先用池扫描列生成构建收敛池（4,969 列）。
   其 LP 松弛值 = 完整池 LP = 191.813620（CG 池收敛证书），故对偶界全局有效（与 03 Benders 同口径）。
2. 子问题按 $rc_p$ 排序取前 K 个负列，单轮 O(P log P)，600 轮约数秒。
3. 次梯度用覆盖次数（实测：布尔标志会导致 λ 卡死在错误点、L≈−5e4；修正后正常收敛）。
4. 停机：600 轮 / ρ<1e-5 / 墙钟 110 s；确定性。


In [1]:
# 环境信息（CONVENTIONS §3.4 要求）
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="numpy")
import sys, platform, math, time
import ortools
sys.path.insert(0, "/mnt/d/exactTest/column-generation-solvers/vrptw_solomon25/scripts")
from cg_cpsat import build_data, enumerate_pool, solve_rmp
from ortools.sat.python import cp_model

print("python", platform.python_version(), "| ortools", ortools.__version__)

n, xc, yc, dem, ready, due, svc, cap, depot_due, dist, d_scaled = build_data()
K = 25
EPS = 1e-7
print(f"客户数 n={n} | 车辆 K={K} | 容量 Q={cap}")
print("已知最优（BKS）: 3 车 / 191.81（两位小数舍入）")


python 3.10.20 | ortools 9.15.6755
python 3.10.20 | ortools 9.15.6755
客户数 n=25 | 车辆 K=25 | 容量 Q=200
已知最优（BKS）: 3 车 / 191.81（两位小数舍入）


In [2]:
# ---- 候选列集：完整池枚举 + 池扫描列生成（精确收敛池）----
t0 = time.time()
full_paths, full_costs, full_masks, _ = enumerate_pool(n, dist, dem, ready, due, svc, cap, depot_due)
path_to_idx = {p: i for i, p in enumerate(full_paths)}
sel = [path_to_idx[(i,)] for i in range(1, n+1)]
sel_set = set(sel)
for it in range(200):
    obj, pi, mu = solve_rmp(n, K, full_paths, full_costs, full_masks, sel)
    add = []
    for p in range(len(full_paths)):
        if p in sel_set:
            continue
        s = 0.0
        mm = full_masks[p]
        while mm:
            lb = mm & -mm
            s += pi[lb.bit_length()-1]
            mm -= lb
        rc = full_costs[p] - s - mu
        if rc < -EPS:
            add.append((rc, p))
    add.sort()
    add = [p for _, p in add[:2000]]
    if not add:
        break
    for p in add:
        if p not in sel_set:
            sel_set.add(p)
            sel.append(p)
print(f"完整池 {len(full_paths)} 列 -> CG 收敛池 {len(sel)} 列 | LP 下界 {round(obj, 6)} | 耗时 {round(time.time()-t0, 1)}s")

paths = [full_paths[p] for p in sel]
costs = [full_costs[p] for p in sel]
masks = [full_masks[p] for p in sel]
P = len(sel)

def rc_of(p, lam):
    s = 0.0
    mm = masks[p]
    while mm:
        lb = mm & -mm
        s += lam[lb.bit_length()-1]
        mm -= lb
    return costs[p] - s

def lagrangian(lam):
    """子问题：取 rc 最小的至多 K 个负列。返回 (L, S)。"""
    neg = sorted((rc_of(p, lam), p) for p in range(P))
    S = [p for rc, p in neg[:K] if rc < -EPS]
    return sum(lam[1:]) + sum(rc_of(p, lam) for p in S), S

def repair(S):
    """贪心修复：给未覆盖客户补最便宜的覆盖列（≤K 列）。"""
    S = list(S)
    covered = [False]*(n+1)
    for p in S:
        mm = masks[p]
        while mm:
            lb = mm & -mm
            covered[lb.bit_length()-1] = True
            mm -= lb
    for i in range(1, n+1):
        if covered[i]:
            continue
        best = None
        for p in range(P):
            if p in S or not ((masks[p] >> i) & 1):
                continue
            if len(S) >= K:
                break
            if best is None or costs[p] < costs[best]:
                best = p
        if best is None:
            return None
        S.append(best)
        mm = masks[best]
        while mm:
            lb = mm & -mm
            covered[lb.bit_length()-1] = True
            mm -= lb
    return S


完整池 210449 列 -> CG 收敛池 4969 列 | LP 下界 191.81362 | 耗时 10.2s


In [3]:
# ---- 次梯度法主循环 ----
lam = [0.0]*(n+1)
rho = 2.0
UB_anchor = sum(2*dist(0, i) for i in range(1, n+1))   # 单客户往返总成本（安全上界）
LAM_MAX = 1000.0
STEP_MAX = 100.0
best_LB = -1e18
best_UB = UB_anchor
best_routes = None
no_improve = 0
wall0 = time.time()
for it in range(600):
    L, S = lagrangian(lam)
    if L > best_LB:
        best_LB = L
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= 30:
            rho /= 2.0
            no_improve = 0
    cnt = [0]*(n+1)
    for p in S:
        mm = masks[p]
        while mm:
            lb = mm & -mm
            cnt[lb.bit_length()-1] += 1
            mm -= lb
    g = [0.0]*(n+1)
    for i in range(1, n+1):
        g[i] = 1.0 - float(cnt[i])       # 次梯度 = 1 - 覆盖次数
    S2 = repair(S)
    if S2 is not None:
        c = sum(costs[p] for p in S2)
        if c < best_UB:
            best_UB = c
            best_routes = [paths[p] for p in S2]
    gnorm2 = sum(g[i]*g[i] for i in range(1, n+1))
    step = 0.0 if gnorm2 <= 1e-12 else min(rho * (UB_anchor - L) / gnorm2, STEP_MAX)
    for i in range(1, n+1):
        lam[i] = min(LAM_MAX, max(0.0, lam[i] + step * g[i]))
    if it % 50 == 0 or it == 599:
        gap = (best_UB-best_LB)/best_LB*100 if best_LB > 1e-9 else float('nan')
        print(f"iter {it+1:4d}: L={L:10.4f} best_LB={best_LB:10.4f} best_UB={best_UB:10.4f} "
              f"gap={gap:.4f}% rho={rho:.4f} lam_max={max(lam):.1f}")
    if rho < 1e-5 or time.time()-wall0 > 110:
        print(f"停机 at iter {it+1}（rho={rho:.2e}）")
        break
print("次梯度墙钟:", round(time.time()-wall0, 1), "s")


iter    1: L=    0.0000 best_LB=    0.0000 best_UB= 1132.1979 gap=nan% rho=2.0000 lam_max=90.6


iter   51: L=   43.9490 best_LB=  116.7527 best_UB= 1132.1979 gap=869.7403% rho=2.0000 lam_max=96.9


iter  101: L=-7134.8090 best_LB=  116.7782 best_UB= 1132.1979 gap=869.5289% rho=1.0000 lam_max=47.0


iter  151: L= -548.3417 best_LB=  149.4910 best_UB=  718.6220 gap=380.7125% rho=0.5000 lam_max=35.2


iter  201: L= -524.7407 best_LB=  179.2354 best_UB=  643.4484 gap=258.9962% rho=0.5000 lam_max=25.0


...

iter  401: L=  191.8136 best_LB=  191.8136 best_UB=  191.8136 gap=0.0000% rho=0.0156 lam_max=21.3


iter  451: L=  191.8136 best_LB=  191.8136 best_UB=  191.8136 gap=0.0000% rho=0.0039 lam_max=21.3


iter  501: L=  191.8136 best_LB=  191.8136 best_UB=  191.8136 gap=0.0000% rho=0.0020 lam_max=21.3


iter  551: L=  191.8136 best_LB=  191.8136 best_UB=  191.8136 gap=0.0000% rho=0.0005 lam_max=21.3


iter  600: L=  191.8136 best_LB=  191.8136 best_UB=  191.8136 gap=0.0000% rho=0.0001 lam_max=21.3
次梯度墙钟: 9.1 s


In [4]:
# ---- 最终 MIP 修复（候选池 CP-SAT）+ 结果对比 ----
model = cp_model.CpModel()
yv = [model.NewBoolVar(f"y{p}") for p in range(P)]
for i in range(1, n+1):
    model.Add(sum(yv[p] for p in range(P) if (masks[p] >> i) & 1) >= 1)
model.Add(sum(yv) <= K)
model.Minimize(sum(int(round(costs[p]*100))*yv[p] for p in range(P)))
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 120
solver.parameters.num_search_workers = 8
t0 = time.time()
st = solver.Solve(model)
routes = [paths[p] for p in range(P) if solver.Value(yv[p])]
exact = 0.0
for r in routes:
    seq = [0] + list(r) + [0]
    exact += sum(dist(seq[i], seq[i+1]) for i in range(len(seq)-1))
print("MIP 修复:", solver.StatusName(st), "| 目标", round(exact, 6), "| 车辆", len(routes), "| 耗时", round(time.time()-t0, 1), "s")
for r in routes:
    print("  路线", r)
print()
print("== 拉格朗日结果 ==")
print("对偶下界 best_LB =", round(best_LB, 6))
print("贪心修复 best_UB =", round(best_UB, 6))
print("对偶 gap =", round((exact - best_LB)/best_LB*100, 6), "%")
print("BKS 对比: 3 车 / 191.81 -> match:",
      len(routes) == 3 and abs(round(exact, 2) - 191.81) < 1e-9)
print("与 01/02/03/05/07 已证最优 191.813620 一致:", abs(exact - 191.813620) < 1e-6)


MIP 修复: OPTIMAL | 目标 191.81362 | 车辆 3 | 耗时 1.1 s
  路线 (13, 17, 18, 19, 15, 16, 14, 12)
  路线 (20, 24, 25, 23, 22, 21)
  路线 (5, 3, 7, 8, 10, 11, 9, 6, 4, 2, 1)

== 拉格朗日结果 ==
对偶下界 best_LB = 191.81362
贪心修复 best_UB = 191.81362
对偶 gap = 0.0 %
BKS 对比: 3 车 / 191.81 -> match: True
与 01/02/03/05/07 已证最优 191.813620 一致: True


## 运行结果与结论

- **对偶下界**：次梯度法约 350 轮收敛到 **191.813620**（= 集合覆盖 LP 松弛值，integrality property 保证）。
- **原始上界**：贪心修复在迭代过程中即达到 191.813620；最终 CP-SAT MIP 修复 OPTIMAL = 191.813620（3 车）。
- **对偶 gap = 0.0%**：下界 = 上界 ⇒ 最优性再次被证明（与 01/02/03/05/07 一致）。
- 松弛覆盖约束后子问题退化为「按 reduced cost 排序取前 K 个负列」，无需调用求解器——覆盖型问题的
  拉格朗日松弛天然高效，是该方法的理想适用场景。
- **注意**：次梯度必须用覆盖次数（$1-\sum_p a_{ip}x_p$）；用布尔覆盖标志会使 λ 卡死在错误点（实测 L≈−5×10⁴）。
- **基准最优值来源**：Solomon c101(25) 文献 BKS（3 车、191.81）；本 notebook 下界=上界=191.813620。
